In [2]:
import altair as alt
import folium
import geopandas as gpd
import google.auth
import pandas as pd

import polars as pl
from great_tables import GT

import world_cup_vars as wc_vars
import D1_prep_trips as D1
import D2_prep_stop_arrivals as D2
import chart_utils

credentials, _ = google.auth.default()

GCS_FILE_PATH = wc_vars.GCS_FILE_PATH

## route visualizations

### Map
1. filter to special service routes
2. map with 2 layers: event vs non-event trips, add layer of the change from baseline
* must change to wide, to be able to plot event vs non-event, otherwise we might get legends to not align and have to set manual cutoffs that will be relevant, will need visual inspection each time

### Stop Arrivals
1. can stops near POI show up on special service routes?

In [3]:
sofi_trips = D1.filter_fct_daily_schedule_rt_route_direction_summary_to_special_routes(
    event_name = wc_vars.event_name,
    operator_list = wc_vars.socal_names,
    route_name_dict = wc_vars.special_socal_routes_dict,
    event_time_of_day_dict = wc_vars.sofi_match_times  
)

In [7]:
sofi_trips_subset = sofi_trips[sofi_trips.schedule_name=="G Trans Schedule"].reset_index(drop=True)

In [8]:
sofi_trips_subset.dtypes

shape_array_key              object
geometry                   geometry
service_date         datetime64[ns]
schedule_name                object
feed_key                     object
route_id                     object
route_id_cleaned             object
route_name                   object
direction_id                  int64
route_type                   object
shape_id                     object
n_trips                       int64
num_stop_times                int64
event_day                      bool
day_type                     object
event_time_of_day            object
dtype: object

In [32]:
trips_by_event = (
    sofi_trips_subset
    .groupby(["schedule_name", "route_name", "direction_id", 
              "route_type", "event_day"])
    .agg(
        total_trips = ("n_trips", "sum"),
        n_days = ("service_date", "nunique"),
    ).reset_index()
)

trips_by_event = trips_by_event.assign(
    daily_trips = trips_by_event.total_trips.divide(trips_by_event.n_days).round(1)
)

In [33]:
# get the most common geom across all service_dates
route_group_cols = [
    "schedule_name", "route_name", "direction_id", 
    "route_type", "event_day"
]
sofi_route_geom = (
    sofi_trips_subset[route_group_cols + ["shape_id", "geometry"]]
    .sort_values(route_group_cols)
    .drop_duplicates(subset=route_group_cols)
    .reset_index(drop=True)
)

In [34]:
trips_by_event_with_geom = pd.merge(
    sofi_route_geom,
    trips_by_event,
    on = route_group_cols,
    how = "inner"
)

In [39]:
trips_by_event_with_geom[trips_by_event_with_geom.event_day == True]

,schedule_name,route_name,direction_id,route_type,event_day,shape_id,geometry,total_trips,n_days,daily_trips
1,G Trans Schedule,7X__7X Line 7X,0,3,True,shp-7X-59,"LINESTRING (-118.34398 33.95246, -118.34432 33...",390,6,65.0
3,G Trans Schedule,7X__7X Line 7X,1,3,True,shp-7X-10,"LINESTRING (-118.28770 33.86904, -118.28758 33...",390,6,65.0


In [48]:
m = trips_by_event_with_geom[trips_by_event_with_geom.event_day == True].explore(
    "daily_trips", 
    tiles = "CartoDB Positron", 
    name = "Event Trips",
    legend=True,
    cmap = "viridis", 
    categorical=False
)

#m = trips_by_event_with_geom[trips_by_event_with_geom.event_day == False].explore(
#    "daily_trips",
#    m=m,
#    name = "Non-Event Trips",
#    legend=True,
#)

folium.LayerControl().add_to(m)

m

In [ ]:
daily_stops = gpd.read_parquet(
    f"{GCS_FILE_PATH}fct_daily_scheduled_stops_{wc_vars.event_name}.parquet",
    storage_options = {"token": credentials}
)
# don't need dim_stops

In [ ]:
daily_stops.dtypes

In [ ]:
stops_gdf = daily_stops[[
    "feed_key", "stop_id", 
    "stop_name", "route_type_array", "geometry"]
    ]

In [ ]:
# how to filter a list column
#daily_stops[(daily_stops.route_type_3 > 0) & 
#    (daily_stops.route_type_array==[2])]